# 02. 관리 서버 설치 — K3s · ArgoCD · Sealed Secrets · Tailscale

| 항목 | 내용 |
|---|---|
| 목표 | 관리 서버에 쿠버네티스(K3s)와 배포 도구를 설치 |
| 소요 시간 | 30~40분 |
| 시작 전 조건 | 01 완료(사설 IP, 인터넷·DNS 정상) |
| 실행 위치 | 🖥 서버 콘솔 (절마다 다시 표시) |

> 대문자 대괄호 `[LIKE_THIS]` 는 **자기 환경의 값으로 바꿔 쓰는 자리표시자**입니다. 표는 [00_overview](00_overview.ipynb)에 있습니다.
> 이 노트북의 셀은 **문서**입니다. 설명대로 해당 위치(서버 콘솔 등)에서 직접 실행하세요. 명령에 따라 서버 설정이 바뀝니다.

## A. 설치 파일 받기

저장소가 **비공개**라 서버에서 `git clone` 하려면 로그인이 필요합니다. 서버에 토큰을 남기지 않도록 **기기 코드 로그인 후 바로 로그아웃**합니다.

1. 서버 콘솔에서 `gh auth login` 을 실행하면 8글자 코드가 나옵니다. (서버에서 브라우저가 안 열려도 괜찮습니다. Enter 를 누르지 말고 기다리세요.)
2. **PC 나 휴대폰 브라우저**에서 `https://github.com/login/device` 를 열어 그 코드를 입력하고 승인합니다.
3. 받자마자 로그아웃합니다.

In [ ]:
!sudo apt install -y gh
!gh auth login -h github.com -p https -w
!gh repo clone [GITHUB_USER]/[REPO] ~/ssm
!gh auth logout -h github.com
!gh auth status                       # "not logged into any GitHub hosts" 가 나와야 함
!ls ~/ssm/deploy/host

> 저장소 이름의 철자를 확인하세요(폴더 이름과 같게 만들었다면 그대로). 틀리면 `Could not resolve to a Repository` 가 나옵니다.

## B. 설치할 버전

스크립트는 버전을 **일부러 필수 값**으로 받습니다(최신을 자동으로 받지 않음). 아래 사이트에서 **안정(prerelease 아님)** 버전을 확인해 적어 두고 쓰세요. 이 안내서를 쓴 시점(2026-11)의 값:

| 구성 | 버전 | 확인 |
|---|---|---|
| K3s | `v1.37.2+k3s1` | github.com/k3s-io/k3s/releases |
| ArgoCD | `v3.6.1` | github.com/argoproj/argo-cd/releases |
| Sealed Secrets | `v0.41.2` | github.com/bitnami-labs/sealed-secrets/releases |

## C. 계획 확인 → 설치

스크립트는 먼저 **인터넷 점검(4곳)** 을 하고 할 일 목록을 보여 준 뒤 멈춥니다(`SSM_CONFIRM=yes` 를 붙여야 실제로 설치).

- 할 일: K3s server 설치(노드 IP 는 사설 IP), ArgoCD·Sealed Secrets 설치, ufw **허용** 규칙만 추가(기본 거부는 건드리지 않음), Tailscale 설치.
- ArgoCD 는 정의 파일(CRD)이 커서 일반 `apply` 가 `Too long: must have at most 262144 bytes` 로 실패합니다 → 스크립트는 **`--server-side`** 로 적용합니다.

In [ ]:
!cd ~/ssm/deploy/host
# 1) 계획만 확인(아직 아무것도 설치하지 않음)
!sudo SSM_PRIVATE_IP=[MGMT_PRIVATE_IP] K3S_VERSION=v1.37.2+k3s1 ARGOCD_VERSION=v3.6.1 SEALED_SECRETS_VERSION=v0.41.2 bash bootstrap-ubuntu01.sh

In [ ]:
# 2) 실제 설치(5~10분)
!sudo SSM_CONFIRM=yes SSM_PRIVATE_IP=[MGMT_PRIVATE_IP] K3S_VERSION=v1.37.2+k3s1 ARGOCD_VERSION=v3.6.1 SEALED_SECRETS_VERSION=v0.41.2 bash bootstrap-ubuntu01.sh

> 스크립트는 여러 번 실행해도 안전합니다(설치된 K3s 는 건너뜀). 중간에 멈추면 같은 명령을 다시 실행하세요.
> 인터넷 점검에서 막히는 주소가 있으면 **설치하지 말고** 01 의 점검으로 돌아가세요.

## D. 확인

In [ ]:
!sudo k3s kubectl get nodes                  # STATUS 가 Ready
!sudo k3s kubectl -n argocd get pods          # 모두 Running (1~3분 걸림)
!sudo k3s kubectl -n kube-system get pods | grep -i sealed
!sudo ufw status numbered | head -20
!ls -ld /var/lib/rancher /etc/rancher/k3s/k3s.yaml     # 700 / 600 (학생 계정이 못 읽게)

## 문제가 생기면
| 증상 | 조치 |
|---|---|
| `Too long: may not be more than 262144 bytes` | ArgoCD 를 `--server-side` 로 다시 적용(스크립트 최신 버전) |
| `curl: (22) … error: 405` 로 멈춤(`ghcr.io`) | 옛 스크립트의 점검 오류. 최신 저장소로 `git pull` |
| `tailscale: command not found` | 설치 단계가 끝까지 안 갔음. `curl -fsSL https://tailscale.com/install.sh \| sh` |
| 한글이 `◆` 로 깨짐 | 서버 콘솔에 한글 글꼴이 없음. 영어 출력 스크립트를 쓰거나 SSH 로 접속 |

## ✅ 완료 확인

- [ ] `k3s kubectl get nodes` 가 Ready
- [ ] argocd 파드가 모두 Running
- [ ] sealed-secrets 파드가 있다
- [ ] 저장소 로그아웃 상태(`gh auth status`)